In [ ]:
# Example 1: Import the main public client classes from azure.search.documents.
from azure.search.documents import SearchClient, SearchItemPaged
from azure.core.credentials import AzureKeyCredential

print("SearchClient:", SearchClient)
print("SearchItemPaged:", SearchItemPaged)
print("AzureKeyCredential:", AzureKeyCredential)

assert SearchClient is not None
assert SearchItemPaged is not None


In [ ]:
# Example 2: Construct a SearchClient with dummy credentials (no network call).
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

client = SearchClient(
    endpoint="https://example.search.windows.net",
    index_name="test-index",
    credential=AzureKeyCredential("fake-key-for-construction-only"),
)

print("client:", client)
# The regenerated (azure-core) client stores endpoint + index_name on its
# _config object rather than the former _endpoint / _index_name attributes.
print("endpoint:", client._config.endpoint)
print("index_name:", client._config.index_name)

assert client._config.index_name == "test-index"
assert client._config.endpoint == "https://example.search.windows.net"
client.close()

In [ ]:
# Example 3: Build IndexAction objects via the IndexDocumentsBatch helper.
from azure.search.documents import IndexDocumentsBatch

batch = IndexDocumentsBatch()
batch.add_upload_actions([{"id": "1", "name": "foo"}, {"id": "2", "name": "bar"}])
batch.add_merge_actions([{"id": "1", "name": "foo-updated"}])
batch.add_delete_actions([{"id": "2"}])

actions = batch.actions
print(f"action count: {len(actions)}")
# IndexAction is now a dict-like model: action_type is an IndexActionType enum and
# the document payload is exposed via as_dict() ('@search.action' + doc fields),
# not the former `additional_properties` attribute.
for a in actions:
    print(" ", a.action_type, a.as_dict())

assert len(actions) == 4
assert actions[0].action_type.value == "upload"
assert actions[0].as_dict()["id"] == "1"

In [ ]:
# Example 4: Inspect models in azure.search.documents.models (query types, etc.).
from azure.search.documents.models import QueryType, IndexAction, IndexActionType

print("QueryType values:")
for member in QueryType:
    print(" ", member.name, "=", member.value)

# IndexAction is a dict-like model keyed by the wire fields, including
# '@search.action' (which drives the action_type enum).
action = IndexAction({"@search.action": "upload", "id": "x", "name": "y"})
print("IndexAction:", action.action_type, action.as_dict())

assert QueryType.SIMPLE.value == "simple"
assert action.action_type == IndexActionType.UPLOAD
assert action.as_dict()["name"] == "y"

In [ ]:
# Example 5: Confirm the async client surface imports + basic shape.
from azure.search.documents.aio import SearchClient as AsyncSearchClient
from azure.core.credentials import AzureKeyCredential

async_client = AsyncSearchClient(
    endpoint="https://example.search.windows.net",
    index_name="test-index",
    credential=AzureKeyCredential("fake-key"),
)

print("async client:", async_client)
print("is coroutine search?", callable(async_client.search))

await async_client.close()

assert async_client._config.index_name == "test-index"
print("PASS: azure_search_documents smoke test complete")